In [3]:
import pandas as pd
from pathlib import Path
import requests
import tarfile
from wikimapper import WikiMapper
from datapackage import Package
import numpy as np

ModuleNotFoundError: No module named 'wikimapper'

In [ ]:
MOVIE_CMU_URL = "http://www.cs.cmu.edu/~ark/personas/data/MovieSummaries.tar.gz"
response = requests.get(MOVIE_CMU_URL, stream=True)
file = tarfile.open(fileobj=response.raw, mode="r|gz")
file.extractall(path='.')

Path("MovieSummaries").rename("data")
data_path = Path("data")

for file_name in ["character.metadata.tsv", "name.clusters.txt", "plot_summaries.txt", "README.txt", "tvtropes.clusters.txt"]:
    file_path = data_path / file_name
    if file_path.exists():
        file_path.unlink()

cmu_cols = ["movie_wikipedia_id", "movie_freebase_id", "movie_title", "movie_release", "movie_revenue", "movie_runtime", "movie_languages", "movie_countries", "movie_genres"]
cmu_df = (pd.read_csv(
    data_path / "movie.metadata.tsv", 
    sep="\t", 
    header=None, 
    names=cmu_cols, 
    usecols=["movie_wikipedia_id", "movie_title", "movie_release", "movie_revenue", "movie_runtime",  "movie_languages", "movie_countries", "movie_genres"])
    .assign(
        movie_release=lambda df: df.movie_release.astype(str).str.slice(0, 4).replace("nan", pd.NA).astype("Int32"),
    )
)

C:\Users\khush\AppData\Local\Temp\ipykernel_39596\917233878.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  file.extractall(path='.')


In [ ]:
cmu_df

,movie_wikipedia_id,movie_title,movie_release,movie_revenue,movie_runtime,movie_languages,movie_countries,movie_genres
0,975900,Ghosts of Mars,2001,14010832.0,98.0,"{""/m/02h40lc"": ""English Language""}","{""/m/09c7w0"": ""United States of America""}","{""/m/01jfsb"": ""Thriller"", ""/m/06n90"": ""Science..."
1,3196793,Getting Away with Murder: The JonBenét Ramsey ...,2000,NaN,95.0,"{""/m/02h40lc"": ""English Language""}","{""/m/09c7w0"": ""United States of America""}","{""/m/02n4kr"": ""Mystery"", ""/m/03bxz7"": ""Biograp..."
2,28463795,Brun bitter,1988,NaN,83.0,"{""/m/05f_3"": ""Norwegian Language""}","{""/m/05b4w"": ""Norway""}","{""/m/0lsxr"": ""Crime Fiction"", ""/m/07s9rl0"": ""D..."
3,9363483,White Of The Eye,1987,NaN,110.0,"{""/m/02h40lc"": ""English Language""}","{""/m/07ssc"": ""United Kingdom""}","{""/m/01jfsb"": ""Thriller"", ""/m/0glj9q"": ""Erotic..."
4,261236,A Woman in Flames,1983,NaN,106.0,"{""/m/04306rv"": ""German Language""}","{""/m/0345h"": ""Germany""}","{""/m/07s9rl0"": ""Drama""}"
...,...,...,...,...,...,...,...,...
81736,35228177,Mermaids: The Body Found,2011,NaN,120.0,"{""/m/02h40lc"": ""English Language""}","{""/m/09c7w0"": ""United States of America""}","{""/m/07s9rl0"": ""Drama""}"
81737,34980460,Knuckle,2011,NaN,96.0,"{""/m/02h40lc"": ""English Language""}","{""/m/03rt9"": ""Ireland"", ""/m/07ssc"": ""United Ki...","{""/m/03bxz7"": ""Biographical film"", ""/m/07s9rl0..."
81738,9971909,Another Nice Mess,1972,NaN,66.0,"{""/m/02h40lc"": ""English Language""}","{""/m/09c7w0"": ""United States of America""}","{""/m/06nbt"": ""Satire"", ""/m/01z4y"": ""Comedy""}"
81739,913762,The Super Dimension Fortress Macross II: Lover...,1992,NaN,150.0,"{""/m/03_9r"": ""Japanese Language""}","{""/m/03_3d"": ""Japan""}","{""/m/06n90"": ""Science Fiction"", ""/m/0gw5n2f"": ..."


In [ ]:
!wikimapper download enwiki-latest --dir data
!wikimapper create enwiki-latest --dumpdir data --target data/index_enwiki-latest.db

2025-11-05 17:05:01,402 - wikimapper.download - INFO - [data\enwiki-latest-page.sql.gz] already exists, skipping downloading [https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-page.sql.gz]!
2025-11-05 17:05:01,402 - wikimapper.download - INFO - [data\enwiki-latest-page_props.sql.gz] already exists, skipping downloading [https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-page_props.sql.gz]!
2025-11-05 17:05:01,402 - wikimapper.download - INFO - [data\enwiki-latest-redirect.sql.gz] already exists, skipping downloading [https://dumps.wikimedia.org/enwiki/latest/enwiki-latest-redirect.sql.gz]!
2025-11-05 17:05:01,728 - wikimapper.processor - INFO - Creating index for [enwiki-latest] in [data/index_enwiki-latest.db]
2025-11-05 17:05:02,043 - wikimapper.processor - INFO - Parsing pages dump
2025-11-05 17:13:30,334 - wikimapper.processor - INFO - Creating database index on 'wikipedia_title'
2025-11-05 17:14:02,279 - wikimapper.processor - INFO - Parsing page properties dump
2025-11

In [ ]:
mapper = WikiMapper(data_path / "index_enwiki-latest.db")
cmu_df = (cmu_df.assign(
            movie_wikidata_id = lambda x: x.movie_wikipedia_id.apply(
                lambda wikipedia_id: mapper.wikipedia_id_to_id(wikipedia_id)
                )
            )
            .drop(columns=["movie_wikipedia_id"])
         )

In [ ]:
WIKI_DATA_SERVICE_URL = 'https://query.wikidata.org/sparql'
query = '''
SELECT DISTINCT ?movie ?book ?bookLabel ?authorLabel ?instanceOfLabel ?countryLabel ?pubDateLabel ?genreLabel ?awardLabel ?seriesLabel ?goodreadsLabel
WHERE 
{
  VALUES ?bookType { wd:Q47461344 wd:Q7725634 wd:Q571 wd:Q14406742 wd:Q21198342 wd:Q277759 }
  VALUES ?movieType { wd:Q11424 wd:Q506240 }

  ?book wdt:P31 ?bookType.
  OPTIONAL {?book wdt:P50 ?author}
  OPTIONAL {?book wdt:P31 ?instanceOf}
  OPTIONAL {?book wdt:P495 ?country}
  OPTIONAL {?book wdt:P577 ?pubDate}
  OPTIONAL {?book wdt:P136 ?genre}
  OPTIONAL {?book wdt:P166 ?award}
  OPTIONAL {?book wdt:P179 ?series}
  OPTIONAL {?book wdt:P8383 ?goodreads}

  ?movie wdt:P31 ?movieType;          
         wdt:P144 ?book.

  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
'''
query_result = requests.get(WIKI_DATA_SERVICE_URL, params = {'format': 'json', 'query': query})
wikidata_df =pd.DataFrame(query_result.json()['results']['bindings'])
for column in wikidata_df.columns:
    wikidata_df[column] = wikidata_df[column].apply(lambda x: x['value'] if isinstance(x, dict) and 'value' in x else x)
  
def get_list(series: pd.Series) -> list:
    return list(set(series.dropna().tolist()))

def mode(x: pd.Series) -> pd.Series:
    modes = x.mode()
    if len(modes) > 0:
        return modes.iloc[0]
    return None

categories = {
    'fiction': {'novel', 'short novel', 'novella', 'serialized fiction', 'short story', 'war fiction', 'magic realist fiction', 'metafiction', 'science fiction', 'suspense in literature', 'horror novel', 'horror fiction', 'crime fiction', 'psychological thriller', 'speculative/fantastic fiction', 'adventure fiction', 'detective fiction', 'noir fiction', 'political novel', 'vampire fiction', 'dystopian fiction', 'social science fiction', 'techno-thriller', 'thriller', 'fantasy', 'Gothic novel', 'picaresque novel', 'mystery fiction', 'post-apocalyptic fiction', 'philosophical fiction', 'romantic fiction', 'Bildungsroman', 'roman à clef', 'comedy', 'black comedy'},
    'non_fiction': {'nonfiction', 'memoir', 'autobiography', 'biographical novel', 'biography', 'essay'},
    'children': {'children\'s literature', 'children\'s fiction', 'young adult fiction', 'children\'s novel'},
    'historical': {'historical fiction', 'historical novel'},
    'drama': {'play', 'drama', 'tragedy'},
    'anime': {'adventure anime and manga', 'drama anime and manga'},
    'fantasy': {'magic realist fiction', 'fantasy', 'vampire fiction', 'fairy tale'},
    'science_fiction': {'science fiction', 'dystopian fiction', 'social science fiction', 'techno-thriller', 'post-apocalyptic fiction'},
    'horror': {'horror novel', 'horror fiction'},
    'thriller': {'psychological thriller', 'thriller'},
    'detective': {'detective fiction', 'noir fiction', 'mystery fiction', 'cloak and dagger novel'},
    'satire': {'satire', 'satirical fiction', 'metafiction'},
    'comedy': {'comedy', 'black comedy'},
}

wikidata_df = (wikidata_df
                .assign(
                    movie_wikidata_id = lambda x: x.movie.str.split('/').str[-1],
                    book_wikidata_id = lambda x: x.book.str.split('/').str[-1],
                    book_release = lambda x: pd.to_datetime(x.pubDateLabel, errors='coerce').dt.year.astype('Int64')
                )
                .groupby(['movie_wikidata_id', 'book_wikidata_id'])
                .agg(
                    book_title = pd.NamedAgg(column='bookLabel', aggfunc=mode),
                    book_author = ('authorLabel', 'first'),
                    book_release = ('book_release', 'first'),
                    book_country = ('countryLabel', 'first'),
                    book_goodreads_id = ('goodreadsLabel', 'first'),
                    series = ('seriesLabel', 'first'),
                    instance_of = pd.NamedAgg(column='instanceOfLabel', aggfunc=get_list),
                    genre = pd.NamedAgg(column='genreLabel', aggfunc=get_list),
                    award = pd.NamedAgg(column='awardLabel', aggfunc=get_list)
                )
                .assign(
                    book_part_of_series = lambda x: x.series.notnull().astype(int),
                    literary_work = lambda x: x.instance_of.apply(lambda y: 'literary work' in y).astype(int),
                    written_work = lambda x: x.instance_of.apply(lambda y: 'written work' in y).astype(int),
                    comic_book_seris = lambda x: x.instance_of.apply(lambda y: 'comic book series' in y).astype(int),
                    book_series = lambda x: x.instance_of.apply(lambda y: 'book series' in y).astype(int),
                    manga_series = lambda x: x.instance_of.apply(lambda y: 'manga series' in y).astype(int),
                    book_fiction = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['fiction'])) > 0).astype(int),
                    book_non_fiction = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['non_fiction'])) > 0).astype(int),
                    book_children = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['children'])) > 0).astype(int),
                    book_historical = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['historical'])) > 0).astype(int),
                    book_drama = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['drama'])) > 0).astype(int),
                    book_anime = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['anime'])) > 0).astype(int),
                    book_fantasy = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['fantasy'])) > 0).astype(int),
                    book_science_fiction = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['science_fiction'])) > 0).astype(int),
                    book_horror = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['horror'])) > 0).astype(int),
                    book_thriller = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['thriller'])) > 0).astype(int),
                    book_detective = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['detective'])) > 0).astype(int),
                    book_satire = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['satire'])) > 0).astype(int),
                    book_comedy = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['comedy'])) > 0).astype(int),
                    book_won_price = lambda x: x.award.apply(lambda y: len(y) > 0).astype(int),
                )
                .drop(['instance_of', 'genre', 'award', 'series'], axis=1)
                .reset_index()
                )

C:\Users\khush\AppData\Local\Temp\ipykernel_29672\418483006.py:59: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  book_release = lambda x: pd.to_datetime(x.pubDateLabel, errors='coerce').dt.year.astype('Int64')


In [ ]:
import time
from typing import List, Dict, Optional
import pandas as pd
import requests

WIKIDATA_SERVICE_URL = 'https://query.wikidata.org/sparql'
HEADERS = {
    'User-Agent': 'MovieBookDatasetCreation/1.0 (khush@illinois.edu) Python/3.9'
}

def run_sparql_query(query: str, max_retries: int = 3) -> Dict:
    """Run a SPARQL query with retry logic and error handling."""
    for attempt in range(max_retries):
        try:
            response = requests.get(
                WIKIDATA_SERVICE_URL,
                params={'format': 'json', 'query': query},
                headers=HEADERS,
                timeout=30
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                raise Exception(f"Failed to query WikiData after {max_retries} attempts: {str(e)}")
            print(f"Attempt {attempt + 1} failed, retrying after 5 seconds...")
            time.sleep(5)

def get_all_books() -> List[str]:
    """Get all book IDs from WikiData matching our criteria."""
    query = '''
    SELECT DISTINCT ?book
    WHERE 
    {
      VALUES ?bookType { wd:Q47461344 wd:Q7725634 wd:Q571 wd:Q14406742 wd:Q21198342 wd:Q277759 }
      ?book wdt:P31 ?bookType.
      SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
    }
    '''
    results = run_sparql_query(query)
    book_df = pd.DataFrame(results['results']['bindings'])
    if book_df.empty:
        return []
    
    for column in book_df.columns:
        book_df[column] = book_df[column].apply(lambda x: x['value'] if isinstance(x, dict) and 'value' in x else x)
    
    return book_df.book.str.split('/').str[-1].tolist()

def get_book_details(ids: List[str]) -> pd.DataFrame:
    """Get detailed information for a list of book IDs."""
    book_ids = " ".join([f"wd:{id}" for id in ids])
    query = '''
    SELECT DISTINCT ?book ?bookLabel ?authorLabel ?instanceOfLabel ?countryLabel 
                    ?pubDateLabel ?genreLabel ?awardLabel ?seriesLabel
    WHERE 
    {
        VALUES ?book { %s }
        OPTIONAL {?book wdt:P50 ?author}
        OPTIONAL {?book wdt:P31 ?instanceOf}
        OPTIONAL {?book wdt:P495 ?country}
        OPTIONAL {?book wdt:P577 ?pubDate}
        OPTIONAL {?book wdt:P136 ?genre}
        OPTIONAL {?book wdt:P166 ?award}
        OPTIONAL {?book wdt:P179 ?series}

        SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
    }
    ''' % book_ids

    results = run_sparql_query(query)
    df = pd.DataFrame(results['results']['bindings'])
    
    if df.empty:
        print(f"No data found for batch of {len(ids)} books")
        return pd.DataFrame()
    
    for column in df.columns:
        df[column] = df[column].apply(lambda x: x['value'] if isinstance(x, dict) and 'value' in x else x)
    
    return df

# Main execution
print("Fetching all book IDs from WikiData...")
book_ids = get_all_books()
print(f"Found {len(book_ids)} books. Processing in batches of 100...")

book_df_list = []
BATCH_SIZE = 100

for i in range(0, len(book_ids), BATCH_SIZE):
    batch = book_ids[i:i + BATCH_SIZE]
    print(f"Processing batch {i//BATCH_SIZE + 1} of {(len(book_ids) + BATCH_SIZE - 1)//BATCH_SIZE} ({len(batch)} books)")
    
    try:
        batch_df = get_book_details(batch)
        if not batch_df.empty:
            book_df_list.append(batch_df)
            print(f"Successfully processed {len(batch_df)} books from this batch")
        
        # Add a small delay to avoid hitting rate limits
        time.sleep(1)
    except Exception as e:
        print(f"Error processing batch {i//BATCH_SIZE + 1}: {str(e)}")
        print(f"Continuing with next batch...")
        continue

# Combine all results
if book_df_list:
    book_df = pd.concat(book_df_list, ignore_index=True)
    print(f"\nFinal dataset contains {len(book_df)} books with detailed information")
    
    # Display first few rows and data info
    print("\nFirst few rows of the dataset:")
    display(book_df.head())
    
    print("\nDataset information:")
    print(book_df.info())
else:
    print("No data was collected. Please check the error messages above.")

Fetching all book IDs from WikiData...
Found 605496 books. Processing in batches of 100...
Processing batch 1 of 6055 (100 books)
Successfully processed 114 books from this batch
Processing batch 2 of 6055 (100 books)
Successfully processed 100 books from this batch
Processing batch 3 of 6055 (100 books)
Successfully processed 147 books from this batch
Processing batch 4 of 6055 (100 books)
Successfully processed 116 books from this batch
Processing batch 5 of 6055 (100 books)
Successfully processed 109 books from this batch
Processing batch 6 of 6055 (100 books)
Successfully processed 112 books from this batch
Processing batch 7 of 6055 (100 books)
Successfully processed 112 books from this batch
Processing batch 8 of 6055 (100 books)
Successfully processed 113 books from this batch
Processing batch 9 of 6055 (100 books)
Successfully processed 108 books from this batch
Processing batch 10 of 6055 (100 books)
Successfully processed 115 books from this batch
Processing batch 11 of 6055 

,book,bookLabel,instanceOfLabel,pubDateLabel,authorLabel,countryLabel,genreLabel,seriesLabel,awardLabel
0,http://www.wikidata.org/entity/Q80083344,"Liber Studiorum: Plate 45, French Beggars",book,NaN,NaN,NaN,NaN,NaN,NaN
1,http://www.wikidata.org/entity/Q80083350,"Liber Studiorum: Plate 46, Sketch after Teniers",book,NaN,NaN,NaN,NaN,NaN,NaN
2,http://www.wikidata.org/entity/Q80083351,"Liber Studiorum: Plate 21, View in North Wales",book,NaN,NaN,NaN,NaN,NaN,NaN
3,http://www.wikidata.org/entity/Q80083348,"Liber Studiorum: Plate 20, Felbrigg Heath, Nor...",book,NaN,NaN,NaN,NaN,NaN,NaN
4,http://www.wikidata.org/entity/Q80083355,"Liber Studiorum: Plate 47, A Study",book,NaN,NaN,NaN,NaN,NaN,NaN



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 721606 entries, 0 to 721605
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   book             721606 non-null  object
 1   bookLabel        721606 non-null  object
 2   instanceOfLabel  721606 non-null  object
 3   pubDateLabel     282497 non-null  object
 4   authorLabel      571140 non-null  object
 5   countryLabel     214712 non-null  object
 6   genreLabel       224453 non-null  object
 7   seriesLabel      34376 non-null   object
 8   awardLabel       14542 non-null   object
dtypes: object(9)
memory usage: 49.5+ MB
None


In [ ]:
book_df = (book_df
                .assign(
                    book_wikidata_id = lambda x: x.book.str.split('/').str[-1],
                    book_release = lambda x: pd.to_datetime(x.pubDateLabel, errors='coerce').dt.year.astype('Int64')
                )
                .groupby(['book_wikidata_id'])
                .agg(
                    book_title = pd.NamedAgg(column='bookLabel', aggfunc=mode),
                    book_author = ('authorLabel', 'first'),
                    book_release = ('book_release', 'first'),
                    book_country = ('countryLabel', 'first'),
                    series = ('seriesLabel', 'first'),
                    instance_of = pd.NamedAgg(column='instanceOfLabel', aggfunc=get_list),
                    genre = pd.NamedAgg(column='genreLabel', aggfunc=get_list),
                    award = pd.NamedAgg(column='awardLabel', aggfunc=get_list)
                )
                .assign(
                    book_part_of_series = lambda x: x.series.notnull().astype(int),
                    literary_work = lambda x: x.instance_of.apply(lambda y: 'literary work' in y).astype(int),
                    written_work = lambda x: x.instance_of.apply(lambda y: 'written work' in y).astype(int),
                    comic_book_seris = lambda x: x.instance_of.apply(lambda y: 'comic book series' in y).astype(int),
                    book_series = lambda x: x.instance_of.apply(lambda y: 'book series' in y).astype(int),
                    manga_series = lambda x: x.instance_of.apply(lambda y: 'manga series' in y).astype(int),
                    book_fiction = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['fiction'])) > 0).astype(int),
                    book_non_fiction = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['non_fiction'])) > 0).astype(int),
                    book_children = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['children'])) > 0).astype(int),
                    book_historical = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['historical'])) > 0).astype(int),
                    book_drama = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['drama'])) > 0).astype(int),
                    book_anime = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['anime'])) > 0).astype(int),
                    book_fantasy = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['fantasy'])) > 0).astype(int),
                    book_science_fiction = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['science_fiction'])) > 0).astype(int),
                    book_horror = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['horror'])) > 0).astype(int),
                    book_thriller = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['thriller'])) > 0).astype(int),
                    book_detective = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['detective'])) > 0).astype(int),
                    book_satire = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['satire'])) > 0).astype(int),
                    book_comedy = lambda x: x.genre.apply(lambda y: len(set(y).intersection(categories['comedy'])) > 0).astype(int),
                    book_won_price = lambda x: x.award.apply(lambda y: len(y) > 0).astype(int),
                )
                .drop(['instance_of', 'genre', 'award', 'series'], axis=1)
                .reset_index()
                ) 

In [ ]:
! mkdir ~/.kaggle
! mv kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

The syntax of the command is incorrect.
'mv' is not recognized as an internal or external command,
operable program or batch file.
'chmod' is not recognized as an internal or external command,
operable program or batch file.
